In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import numpy as np
import json
import torch
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

from sklearn.metrics import classification_report

In [ ]:
import os
base_path = None

for root, dirs, files in os.walk("/kaggle/input"):
    if "best_preds.npy" in files:
        base_path = root
        break

print("Detected path:", base_path)

In [ ]:
import numpy as np
import json
import torch

preds = np.load(os.path.join(base_path, "best_preds.npy"))
labels = np.load(os.path.join(base_path, "best_labels.npy"))
conf_matrix = np.load(os.path.join(base_path, "confusion_matrix.npy"))
confidences = np.load(os.path.join(base_path, "confidences.npy"))

with open(os.path.join(base_path, "history.json")) as f:
    history = json.load(f)

with open(os.path.join(base_path, "per_class_acc.json")) as f:
    per_class_acc = json.load(f)

class_map = torch.load(os.path.join(base_path, "class_map_50.pth"))

In [ ]:
accuracy = (preds == labels).mean()

print(f"Overall Accuracy: {accuracy*100:.2f}%")
print(f"Total Samples: {len(labels)}")

In [ ]:
print(classification_report(labels, preds))

In [ ]:
plt.figure(figsize=(12,10))
sns.heatmap(conf_matrix, cmap="Blues")
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()

In [ ]:
plt.plot(history["train_loss"], label="Train Loss")
plt.plot(history["val_loss"], label="Validation Loss")

plt.title("Loss Curve")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()

In [ ]:
plt.plot(history["train_acc"], label="Train Accuracy")
plt.plot(history["val_acc"], label="Validation Accuracy")

plt.title("Accuracy Curve")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.show()

In [ ]:
plt.plot(history["top3_acc"], label="Top-3 Accuracy")
plt.plot(history["top5_acc"], label="Top-5 Accuracy")

plt.title("Top-K Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.show()

In [ ]:
plt.plot(history["f1_macro"], label="F1 Macro")
plt.plot(history["f1_weighted"], label="F1 Weighted")

plt.title("F1 Score")
plt.xlabel("Epoch")
plt.ylabel("Score")
plt.legend()
plt.show()

In [ ]:
plt.hist(confidences, bins=50)
plt.title("Prediction Confidence Distribution")
plt.xlabel("Confidence")
plt.ylabel("Frequency")
plt.show()

In [ ]:
df = pd.DataFrame({
    "Class": list(range(len(per_class_acc))),
    "Accuracy": per_class_acc
})

df_sorted = df.sort_values(by="Accuracy")

print("Worst Classes:")
print(df_sorted.head())

print("\nBest Classes:")
print(df_sorted.tail())

In [ ]:
plt.figure(figsize=(10,8))
plt.bar(df_sorted["Class"], df_sorted["Accuracy"])
plt.title("Per-Class Accuracy")
plt.xlabel("Accuracy")
plt.ylabel("Class")
plt.show()

In [ ]:
wrong_idx = np.where(preds != labels)[0]

print(f"Total wrong predictions: {len(wrong_idx)}")

for i in wrong_idx[:10]:
    print(f"True: {labels[i]} | Predicted: {preds[i]}")

In [ ]:
inv_class_map = {v:k for k,v in class_map.items()}

# example
for i in range(5):
    print(f"{i} → {inv_class_map[i]}")

In [ ]:
import torch
import numpy as np
import pandas as pd
import os
import cv2

from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from transformers import VideoMAEForVideoClassification

# ================= CONFIG =================
NUM_FRAMES = 16
BATCH_SIZE = 4

BEST_MODEL_PATH = "/kaggle/input/models/sarcastttttttic/hehehe/pytorch/default/1/best_model_50.pth"
SELECTED_CLASSES_PATH = "/kaggle/input/datasets/sarcastttttttic/sign50result/selected_classes_50.pth"
CLASS_MAP_PATH = "/kaggle/input/datasets/sarcastttttttic/sign50result/class_map_50.pth"

TEST_VIDEO_DIR = "/kaggle/input/datasets/sttaseen/autsl/test"
TEST_LABELS_PATH = "/kaggle/input/datasets/sttaseen/autsl/test_labels.csv"

# ================= LOAD CLASS INFO =================
selected_classes = torch.load(SELECTED_CLASSES_PATH)
class_map = torch.load(CLASS_MAP_PATH)

# ================= LOAD LABELS =================
def load_labels(csv_path):
    df = pd.read_csv(csv_path, header=None)
    df.columns = ["video_name", "label"]
    return dict(zip(df["video_name"], df["label"]))

test_labels_full = load_labels(TEST_LABELS_PATH)

test_labels = {
    k: class_map[v]
    for k, v in test_labels_full.items()
    if v in selected_classes
}

num_classes = len(class_map)
print("Using classes:", num_classes)

# ================= DATASET =================
class AUTSLDataset(Dataset):
    def __init__(self, video_dir, labels_dict, transform=None):
        self.video_dir = video_dir
        self.samples = list(labels_dict.items())
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def load_video(self, path):
        cap = cv2.VideoCapture(path)
        frames = []

        while True:
            ret, frame = cap.read()
            if not ret:
                break
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(frame)

        cap.release()

        if len(frames) == 0:
            return None

        if len(frames) < NUM_FRAMES:
            frames = frames * (NUM_FRAMES // len(frames) + 1)

        idxs = np.linspace(0, len(frames)-1, NUM_FRAMES).astype(int)
        return [frames[i] for i in idxs]

    def __getitem__(self, idx):
        video_name, label = self.samples[idx]
        path = os.path.join(self.video_dir, f"{video_name}_color.mp4")

        frames = self.load_video(path)

        if frames is None:
            return self.__getitem__(np.random.randint(0, len(self.samples)))

        frames = [self.transform(frame) for frame in frames]
        return torch.stack(frames), label

# ================= TRANSFORM =================
test_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

test_dataset = AUTSLDataset(TEST_VIDEO_DIR, test_labels, test_transform)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# ================= MODEL =================
model = VideoMAEForVideoClassification.from_pretrained(
    "MCG-NJU/videomae-base",
    num_labels=num_classes,
    ignore_mismatched_sizes=True
)

model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location="cpu"))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

# ================= EVALUATION =================
total = 0
top1_correct = 0
top3_correct = 0
top5_correct = 0

with torch.no_grad():
    for videos, labels in tqdm(test_loader):
        videos, labels = videos.to(device), labels.to(device)

        outputs = model(videos).logits

        preds = outputs.argmax(dim=1)
        top3 = torch.topk(outputs, 3, dim=1).indices
        top5 = torch.topk(outputs, 5, dim=1).indices

        top1_correct += (preds == labels).sum().item()
        total += labels.size(0)

        for i in range(labels.size(0)):
            if labels[i] in top3[i]:
                top3_correct += 1
            if labels[i] in top5[i]:
                top5_correct += 1

# ================= RESULTS =================
print("\n===== TEST RESULTS =====")
print(f"Top-1 Accuracy: {top1_correct/total:.4f}")
print(f"Top-3 Accuracy: {top3_correct/total:.4f}")
print(f"Top-5 Accuracy: {top5_correct/total:.4f}")